# QMUL-SurvFace-v1 공식 데이터 준비

`training_set`을 identity-disjoint development/calibration으로 분리하고, 이와 완전히 별도로 공식 gallery, mated probe, unmated probe 매니페스트를 만듭니다. PCA/PQ는 training development에서만 fit하고 공식 test에는 fit하지 않습니다. `real, 1.0`은 공식 test 행과 `protocol_index`를 그대로 유지합니다. 축소 실행은 gallery/mated probe를 registered identity 단위로 함께 선택합니다. Unmated probe는 공식 identity label이 없으므로 opaque per-image key 단위로만 결정론적으로 축소합니다. 각 역할의 원래 순서는 `source_protocol_index`에 남기고 실행용 index만 연속으로 다시 부여합니다.

| 모드 | 예상 시간 | 저장 |
| --- | ---: | --- |
| `WRITE_OUTPUTS=False` | 약 20초~1분 | 없음(전체 검증만 수행) |
| `WRITE_OUTPUTS=True` | 약 20초~2분 | `data/interim/survface/`에 원자적으로 저장 |

> **진행/체크포인트/재시작**: 첫 설정 셀의 `MODE`, `DATA_FRACTION`, `SEED`만 바꾸고 Kernel Restart 후 처음부터 실행합니다. 저장 중 중단되면 출력 파일을 확인하고, 의도적으로 다시 만들 때만 `OVERWRITE=True`로 바꿉니다. `training_set`은 공식 test와 섞지 않습니다.


In [ ]:
from __future__ import annotations

from dataclasses import replace
import json
import os
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
            return candidate
    raise FileNotFoundError("D:/ronbun 내부에서 노트북을 실행하십시오.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from time import perf_counter

import numpy as np
import pandas as pd
import scipy
from IPython.display import display

from research.compression import PCA_SWEEP_DIMENSIONS
from research.datasets import (
    build_survface_official_manifest,
    build_survface_training_manifest,
    write_survface_official_bundle,
    write_survface_training_bundle,
)
from research.experiments.scope import (
    ExperimentScope,
    select_manifest_fraction,
    select_open_set_protocol_fraction,
)
from research.protocols import (
    build_survface_official_protocol,
    validate_identity_disjoint_splits,
)

print(f"project_root: {PROJECT_ROOT}")
print(f"python: {sys.executable}")
print(f"pandas: {pd.__version__}, scipy: {scipy.__version__}")


## 1. 경로와 저장 모드

원본 루트에는 `Face_Identification_Evaluation`과 `Face_Identification_Test_Set`이 모두 있어야 합니다. `MODE=dev`와 작은 `DATA_FRACTION`은 실행 점검용이며, `MODE=real`, `DATA_FRACTION=1.0`만 공식 전체 결과입니다.


In [ ]:
# Step 1 실행 범위: 이 셀의 세 값만 바꾸고 Kernel Restart -> Run All
MODE = "dev"             # "dev" 또는 "real"
DATA_FRACTION = 1.0       # 0 < DATA_FRACTION <= 1
SEED = 42

WRITE_OUTPUTS = True
OVERWRITE = True

PCA_DIMENSIONS = (384, 256, 128, 64, 32)
PQ_SOURCE_DIMENSION = 512
TRAINING_DEVELOPMENT_FRACTION = 0.80
if PCA_DIMENSIONS != tuple(PCA_SWEEP_DIMENSIONS):
    raise RuntimeError("노트북 PCA sweep과 research.compression 정의가 다릅니다.")
EXPERIMENT_SCOPE = ExperimentScope(
    mode=MODE, data_fraction=DATA_FRACTION, seed=SEED
)

SURVFACE_ROOT = PROJECT_ROOT / "data" / "raw" / "QMUL-SurvFace"
OUTPUT_DIR = PROJECT_ROOT / "data" / "interim" / "survface"

display(pd.Series({
    "WRITE_OUTPUTS": WRITE_OUTPUTS,
    "OVERWRITE": OVERWRITE,
    "SURVFACE_ROOT": str(SURVFACE_ROOT),
    "OUTPUT_DIR": str(OUTPUT_DIR),
    **EXPERIMENT_SCOPE.as_dict(),
    "PCA_DIMENSIONS": PCA_DIMENSIONS,
    "PQ_SOURCE_DIMENSION": PQ_SOURCE_DIMENSION,
    "TRAINING_DEVELOPMENT_FRACTION": TRAINING_DEVELOPMENT_FRACTION,
}, name="value").to_frame())


## 2. Training split과 공식 test를 분리 검증

먼저 `training_set` identity를 development/calibration으로 나누고 각 split 안에서 scope를 적용합니다. 그 다음 공식 test bundle을 별도로 검증합니다. Gallery와 mated probe는 같은 registered identity 집합으로 함께 선택합니다. Unmated는 실제 identity label이 없으므로 별도 역할에서 opaque per-image key로 선택합니다. 공식 test 행은 PCA/PQ fit 또는 threshold calibration 표본에 들어가지 않습니다.


In [ ]:
started = perf_counter()
print("[RUNNING] SurvFace training_set identity-disjoint split 검증 시작")
full_training_bundle = build_survface_training_manifest(
    SURVFACE_ROOT,
    PROJECT_ROOT,
    seed=SEED,
    development_fraction=TRAINING_DEVELOPMENT_FRACTION,
)
selected_training_development = select_manifest_fraction(
    full_training_bundle.manifest.loc[
        full_training_bundle.manifest["split"].eq("development")
    ],
    EXPERIMENT_SCOPE,
    namespace="survface:training:development",
)
selected_training_calibration = select_manifest_fraction(
    full_training_bundle.manifest.loc[
        full_training_bundle.manifest["split"].eq("calibration")
    ],
    EXPERIMENT_SCOPE,
    namespace="survface:training:calibration",
)
selected_training_ids = set(pd.concat(
    [selected_training_development, selected_training_calibration],
    ignore_index=True,
)["image_id"].astype(str))
selected_training_manifest = full_training_bundle.manifest.loc[
    full_training_bundle.manifest["image_id"].astype(str).isin(selected_training_ids)
].copy().reset_index(drop=True)
validate_identity_disjoint_splits(selected_training_manifest)
if len(selected_training_development) < max(PCA_DIMENSIONS):
    raise ValueError(
        "선택된 SurvFace development 이미지 수가 PCA-384 학습에 부족합니다. "
        "DATA_FRACTION을 늘리십시오."
    )
training_summary = {
    **full_training_bundle.summary,
    "scope": EXPERIMENT_SCOPE.as_dict(),
    "source_image_count": int(len(full_training_bundle.manifest)),
    "image_count": int(len(selected_training_manifest)),
    "identity_count": int(selected_training_manifest["identity_id"].nunique()),
    "development_identity_count": int(
        selected_training_development["identity_id"].nunique()
    ),
    "calibration_identity_count": int(
        selected_training_calibration["identity_id"].nunique()
    ),
    "development_image_count": int(len(selected_training_development)),
    "calibration_image_count": int(len(selected_training_calibration)),
    "official_test_included": False,
}
training_bundle = replace(
    full_training_bundle,
    manifest=selected_training_manifest,
    summary=training_summary,
)
display(pd.Series(training_bundle.summary, name="training").to_frame())
display(training_bundle.manifest.groupby("split").agg(
    images=("image_id", "size"), identities=("identity_id", "nunique")
))

print("[RUNNING] 공식 MAT와 test 이미지 파일 집합 검증 시작")
full_bundle = build_survface_official_manifest(SURVFACE_ROOT, PROJECT_ROOT)

for role, frame in (
    ("gallery", full_bundle.gallery),
    ("registered_probe", full_bundle.registered_probes),
    ("unknown_unknown_probe", full_bundle.unknown_unknown_probes),
):
    actual = frame["protocol_index"].to_numpy(dtype=np.int64)
    expected = np.arange(len(frame), dtype=np.int64)
    if not np.array_equal(actual, expected):
        raise ValueError(f"{role} protocol_index가 공식 순서를 보존하지 않습니다.")
    print(f"[OK] {role}: rows={len(frame):,}, elapsed={perf_counter() - started:.1f}s")

if int(full_bundle.summary["known_unknown_identity_count"]) != 0:
    raise ValueError("SurvFace 공식 프로토콜의 known_unknown은 반드시 0이어야 합니다.")

full_protocol = build_survface_official_protocol(full_bundle.manifest)
selected_protocol = select_open_set_protocol_fraction(
    full_protocol,
    EXPERIMENT_SCOPE,
    namespace="survface:official",
)

def with_execution_index(frame: pd.DataFrame) -> pd.DataFrame:
    selected = frame.copy().reset_index(drop=True)
    selected["source_protocol_index"] = selected["protocol_index"].astype(int)
    selected["protocol_index"] = np.arange(len(selected), dtype=np.int64)
    return selected

selected_gallery = with_execution_index(selected_protocol.gallery)
selected_registered = with_execution_index(selected_protocol.registered_probes)
selected_unknown = with_execution_index(selected_protocol.unknown_unknown_probes)
selected_manifest = pd.concat(
    [selected_gallery, selected_registered, selected_unknown],
    ignore_index=True,
)
combined_boundary_manifest = pd.concat(
    [selected_training_manifest, selected_manifest],
    ignore_index=True,
    sort=False,
)
validate_identity_disjoint_splits(combined_boundary_manifest)
if combined_boundary_manifest["image_id"].duplicated().any():
    raise ValueError("SurvFace training과 official test image_id가 겹칩니다.")
selected_gallery_ids = tuple(sorted(
    selected_gallery["identity_id"].astype(str).unique()
))
selected_unknown_ids = tuple(
    selected_unknown["identity_id"].astype(str).tolist()
)
summary = {
    **full_bundle.summary,
    "scope": EXPERIMENT_SCOPE.as_dict(),
    "source_image_count": int(len(full_bundle.manifest)),
    "image_count": int(len(selected_manifest)),
    "gallery_image_count": int(len(selected_gallery)),
    "registered_probe_image_count": int(len(selected_registered)),
    "unknown_unknown_probe_image_count": int(len(selected_unknown)),
    "registered_identity_count": len(selected_gallery_ids),
    "unmated_probe_count": int(len(selected_unknown)),
    "unmated_sampling_unit": "opaque_per_image_key_no_identity_labels",
    "protocol_index_policy": (
        "official" if EXPERIMENT_SCOPE.is_full_dataset
        else "source_protocol_index_preserved_execution_index_rebased"
    ),
}
bundle = replace(
    full_bundle,
    manifest=selected_manifest,
    gallery_identities=selected_gallery_ids,
    unknown_unknown_identities=selected_unknown_ids,
    summary=summary,
)

display(pd.Series(bundle.summary, name="value").to_frame())
display(bundle.manifest.groupby(["protocol_role", "probe_type"], sort=False).agg(
    images=("image_id", "size"), identities=("identity_id", "nunique")
))
print(f"[COMPLETED] 공식 검증 및 role-aware scope 선택 통과, elapsed={perf_counter() - started:.1f}s")


## 3. 역할별 파일 저장

`official_manifest.csv`는 선택된 전체 역할을 포함합니다. `source_protocol_index`는 공식 순서를 추적하고 `protocol_index`는 선택된 실행 안에서 연속입니다. `real, 1.0`에서는 두 값이 같습니다.


In [ ]:
output_names = (
    "training_manifest.csv",
    "training_summary.json",
    "official_manifest.csv",
    "gallery.csv",
    "registered_probes.csv",
    "unknown_unknown_probes.csv",
    "gallery_identities.txt",
    "unknown_unknown_identities.txt",
    "summary.json",
)

if WRITE_OUTPUTS:
    training_paths = write_survface_training_bundle(
        training_bundle, OUTPUT_DIR, overwrite=OVERWRITE
    )
    official_paths = write_survface_official_bundle(
        bundle, OUTPUT_DIR, overwrite=OVERWRITE
    )
    written_paths = {**training_paths, **official_paths}
    print("[COMPLETED] training 및 공식 test bundle 저장 완료")
else:
    written_paths = {name: OUTPUT_DIR / name for name in output_names}
    print("[REVIEW] WRITE_OUTPUTS=False: 검증만 완료했으며 파일을 저장하지 않았습니다.")

display(pd.DataFrame([
    {"file": name, "path": str(path), "exists": path.is_file()}
    for name, path in written_paths.items()
]))


## 다음 단계

`training_summary.json`의 `official_test_included=false`와 development/calibration identity 분리를 먼저 확인합니다. 이어 `summary.json`의 `scope`, known unknown 0, `protocol_index_policy`를 확인합니다. Step 1 기준 설정은 `configs/experiments/step1_embedding_compression.yaml`이며, 공식 논문 결과에서는 `real, 1.0`과 전체 gallery identity를 사용하는 `official_all` 정책을 유지해야 합니다.
